# Predicción del fútbol uruguayo: ID3 propio y Naive Bayes

En este notebook entrenamos dos clasificadores implementados a mano para
predecir si un partido de primera división lo gana el local (L), termina en
empate (E) o lo gana el visitante (V). Se usan partidos del campeonato
uruguayo y atributos basados solo en lo que pasó antes del partido.

El recorrido es: primero los datos y los atributos, después la forma de elegir
hiperparámetros con validación temporal, y recien al final la medición sobre el
test de 2024-2025. La implementación de cada modelo esta en `src/` (`id3.py`
para el árbol y `naive_bayes.py` para el Naive Bayes).

## 1. Configuración

Se importan las implementaciones compartidas de src/. El flujo de selección
y ajuste es el mismo de entrega/notebook.ipynb; el test queda desactivado.

### Reproducción del informe final

Entorno verificado: Python 3.12.14 y scikit-learn 1.9.1; semilla 42.
Instalar con Python 3.12: `python3.12 -m pip install numpy pandas matplotlib scikit-learn==1.9.1 ipykernel`.
En el repositorio también puede usarse `python3.12 -m pip install -r requirements.txt`.
Abrir este notebook con ese kernel, comprobar `RAW`, cambiar
`RUN_FINAL_TEST = True` en la primera celda de código y ejecutar todas las
celdas en orden desde un kernel nuevo. `False` ejecuta únicamente selección
y ajuste; no reproduce las tablas finales. Las últimas celdas reproducen
métricas, matrices, ejemplos y escenarios del informe sin leer resultados precalculados.
El test ya había sido inspeccionado en versiones anteriores: esta reproducción
no constituye una evaluación independiente sobre datos nuevos.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.naive_bayes import CategoricalNB
from sklearn.tree import DecisionTreeClassifier

# Carpeta con los modulos propios; no se instalan como paquete.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'requirements.txt').is_file() and (p / 'src').is_dir())
sys.path.insert(0, str(ROOT / 'src'))

from baseline import BASELINE_FEATURES, TenYearWinRateClassifier  # noqa: E402
from evaluation import (  # noqa: E402
    evaluate_temporal_cv,
    make_temporal_folds,
    new_discretizer,
    reporte_por_clases,
    temporal_holdout,
)
from features import (  # noqa: E402
    NUMERIC_FEATURES,
    build_causal_match_features,
    load_clean_matches,
    predict_new_match,
)
from id3 import ID3  # noqa: E402
from naive_bayes import MEstimateCategoricalNB  # noqa: E402
from preprocessing import MixedTypeDiscretizer  # noqa: E402

RAW = ROOT / 'data/raw/futbol_uruguayo.zip'
CLASSES = ['E', 'L', 'V']
RANDOM_STATE = 42
RUN_FINAL_TEST = False
pd.set_option('display.max_columns', 30)


def entropia(series):
    """Entropía de Shannon en bits de una serie de etiquetas."""
    probs = series.value_counts(normalize=True)
    return float(-(probs * np.log2(probs)).sum())

## 2. Datos: lectura y limpieza del ZIP

`load_clean_matches` extrae el ZIP original en memoria, normaliza las columnas,
quita partidos duplicados y dudosos, y deriva la clase `winner` exclusivamente
de los goles (`gh`, `ga`): `L` si ganó el local, `V` si ganó el visitante, `E`
si empataron. Los goles nunca son atributos del modelo: solo sirven para la
etiqueta y para los historiales de partidos anteriores.

In [ ]:
matches = load_clean_matches(RAW)
print('Filas limpias:', len(matches))
display(matches.head(3))
display(matches.dtypes.to_frame(name='tipo'))
print('Distribución de la clase (todo el histórico):')
display(
    matches['winner']
    .value_counts()
    .reindex(CLASSES)
    .rename('cantidad')
    .to_frame()
)

## 3. Atributos causales

`build_causal_match_features` calcula, para cada partido, tasas históricas con
fechas estrictamente anteriores. El calculo de atributos procesa primero todos
los partidos de un mismo día y recien después actualiza los historiales, de
modo que un partido jamás usa el resultado de otro partido del mismo día (no
hay leakage).

Se usan seis tasas de victoria que cubren horizontes distintos:

- `home/away_win_rate_last_5`: forma reciente de cada equipo (5 partidos);
- `home/away_win_rate_season`: éxito dentro del año calendario;
- `home_win_rate_as_home_all`: histórico del local cuando juega de local;
- `home_win_rate_h2h_as_home`: histórico del local contra ese rival.

Con eso el modelo puede usar tanto el momento de forma actual como el
desempeño de largo plazo (localia y duelo directo). Los nombres de los equipos
y el mes no entran como atributos: son identificadores que no generalizan y
provocarian sobreajuste.

Si un equipo no tiene historial se imputa un neutro de `0.5` (no `0.0`), para
no confundir "nunca jugó" con "siempre perdió".

In [ ]:
featured = build_causal_match_features(matches)
print('Partidos con atributos:', len(featured))
display(featured[['date', 'home', 'away'] + NUMERIC_FEATURES + ['winner']].head(5))

sin_historial = featured[featured['home_win_rate_season'].eq(0.5)]
print('Ejemplos con tasa 0.5 (neutro imputado o tasa real):')
display(sin_historial[['date', 'home', 'away'] + NUMERIC_FEATURES].head(3))

## 4. Partición temporal y folds

Los datos son series temporales y el uso real es predecir partidos futuros,
así que la partición es temporal y nunca aleatoria (una partición aleatoria
filtraría información del futuro hacia el entrenamiento):

- Train: partidos hasta 2023 inclusive.
- Test: partidos de 2024 y 2025 (queda reservado para la medición final).

Para elegir hiperparámetros se usan los años completos 2021, 2022 y 2023 como
validación, entrenando en cada fold con todos los partidos anteriores a ese
año. El cálculo de selección no usa test; el bloque ya fue inspeccionado en
versiones anteriores y no es un holdout intacto.

In [ ]:
train, test = temporal_holdout(featured)
assert train['date'].dt.year.le(2023).all()
assert test['date'].dt.year.isin([2024, 2025]).all()
print('Train:', len(train), 'partidos (hasta 2023)')
print('Test: ', len(test), 'partidos (2024-2025)')

print('Distribución de clases en train:')
display(train['winner'].value_counts(normalize=True).reindex(CLASSES))

folds = make_temporal_folds(train)
display(pd.DataFrame([
    {'validación': fold.validation_year,
     'train_hasta': train.iloc[list(fold.train_positions)]['date'].max().date(),
     'validacion_desde': train.iloc[list(fold.validation_positions)]['date'].min().date(),
     'validacion_hasta': train.iloc[list(fold.validation_positions)]['date'].max().date(),
     'n_train': len(fold.train_positions),
     'n_validacion': len(fold.validation_positions)}
    for fold in folds
]))

## 5. Discretización

ID3 y Naive Bayes categorico trabajan con enteros. Cada tasa se convierte en
una de tres categorías (`baja`, `media`, `alta`), codificadas 1, 2, 3 (el 0
queda reservado para valores desconocidos).

Dos estrategias de cortes:

- `win_rate_last_5`: cortes fijos `[0.3, 0.6]`. Con cinco antecedentes, significan
  0-1 triunfos, 2 triunfos y 3+ triunfos. Con menos antecedentes se usa la
  proporción sobre los disponibles; sin antecedentes se imputa 0.5. No dependen de los datos y no requieren ajustarse con train.
- El resto: cuantiles (equal-frequency) calculados solo con train.

In [ ]:
discretizer = new_discretizer()
discretizer.fit(train[NUMERIC_FEATURES])

print('Bordes de cada atributo (dos cortes -> tres bines):')
display(
    pd.DataFrame(
        {column: discretizer.numeric_edges_[column] for column in NUMERIC_FEATURES}
    ).T.rename(columns={0: 'borde_1', 1: 'borde_2'})
)

X_train = discretizer.transform(train[NUMERIC_FEATURES])

print('Ejemplo: la tasa cruda y el código que recibe el modelo:')
muestra = train[['home', 'away'] + NUMERIC_FEATURES].head(5).copy()
codigos = pd.DataFrame(
    X_train[:5], columns=[column + '_cod' for column in NUMERIC_FEATURES]
)
display(pd.concat([muestra, codigos], axis=1))

print('Filas de train por categoría:')
resumen = pd.DataFrame({column: pd.Series(X_train[:, i]).map(
    {1: 'baja', 2: 'media', 3: 'alta'}
).value_counts() for i, column in enumerate(NUMERIC_FEATURES)})
display(resumen.reindex(['baja', 'media', 'alta']).fillna(0).astype(int))

## 6. Cómo decide el árbol: entropía y ganancia

ID3 elige el atributo que más reduce la entropia (desorden) de la clase. La
entropia se mide en bits:

```text
H(Y) = -sum_c P(c) * log2(P(c))
```

La ganancia de información de un atributo es la entropia menos la entropia
condicional promedio tras partir por ese atributo. Acá calculamos la primera
división que haria el árbol sobre todo el train.

In [ ]:
parent_entropy = entropia(train['winner'])
print(f'Entropía de la clase en train: {parent_entropy:.4f} bits '
      '(máximo teórico ~1.585 para 3 clases equiprobables)')

gains = []
for index, column in enumerate(NUMERIC_FEATURES):
    conditional = 0.0
    for codigo in np.unique(X_train[:, index]):
        mask = X_train[:, index] == codigo
        conditional += mask.mean() * entropia(train['winner'][mask])
    gains.append({
        'atributo': column,
        'entropia_condicional': round(conditional, 4),
        'ganancia': round(parent_entropy - conditional, 4),
    })
gains_frame = pd.DataFrame(gains).sort_values('ganancia', ascending=False)
display(gains_frame.reset_index(drop=True))

mejor = gains_frame.iloc[0]
print(f'Primera división sobre {mejor["atributo"]} '
      f'(ganancia {mejor["ganancia"]:.4f} bits).')

## 7. Cómo se miden los resultados

Definimos acá, brevemente, las medidas que usamos de aquí en adelante.

**Accuracy** (exactitud): fracción de partidos bien clasificados sobre el total.
Es fácil de entender pero sola engana cuando hay clases desbalanceadas: acá,
predecir siempre L ya rondaria el 40% (L es la clase más comun).

Para el resto pensamos clase por clase. Con tres clases, para cada una (por
ejemplo L) separamos las predicciones en:

- **true positives (TP)**: partidos que eran L y se predijeron L;
- **false positives (FP)**: partidos que no eran L pero se predijeron L;
- **false negatives (FN)**: partidos que eran L y se predijeron otra cosa.

Y se definen:

- **Precision**: TP / (TP + FP). De lo que el modelo predijo como L, que
  fracción era L en realidad.
- **Recall**: TP / (TP + FN). De los L que había, cuántos atrapó.
- **F1-score**: media armónica de precision y recall, `2PR/(P+R)`.
- **Support**: cuántos ejemplos reales de esa clase había.

Como hay tres clases, los promedios se hacen de dos formas:

- **macro** (simple): las tres clases pesan igual; es estricto con la clase
  chica (el empate).
- **weighted** (ponderado): pesa cada clase según su support; refleja el
  dataset real pero puede esconder que una clase chica rinde mal.

El **macro-F1** es el promedio macro del F1 y es el criterio que usamos para
elegir hiperparámetros: con accuracy, un modelo que nunca predice empates
pareceria aceptable. El **error** es `1 - accuracy`, la fracción de partidos
equivocados, y es lo que graficamos como "error de validación".

## 8. ID3: elegir la poda con validación

El hiperparámetro que ajustamos de ID3 es `min_info_gain`: se corta la
expansion cuando la mejor ganancia de información disponible es menor o igual
que ese umbral. Sin poda el árbol crece hasta memorizar ruido del train; el
umbral controla que tan fino sigue creciendo.

Para elegirlo barremos una grilla y medimos con la validación temporal:

- **error_validacion** = `1 - accuracy` y **macro_f1** (sección 7): cada fila
  es la media sobre los tres folds.
- Las columnas `*_std` son la desviación estándar entre esos tres folds. La
  reportamos porque la media sola no alcanza: una desviación chica indica que
  el resultado se mantiene entre temporadas y no depende de un solo año.
- **error_train** es el error sobre el train de cada fold, como control de
  sobreajuste: si es mucho menor que el de validación, el árbol memorizo datos
  de train que no generalizan.

Elegimos el umbral que maximiza el **macro-F1** de validación (da el mismo peso
a E, L y V; con accuracy un árbol que solo predice L quedaria bien ranqueado).
Como desempate, ante valores iguales tomamos el umbral menor: podamos solo
cuando la validación lo justifica. En la tabla el pico es claro, `0.005`:
después el macro-F1 no mejora.

In [ ]:
grid_gain = [0.0, 0.001, 0.005, 0.01, 0.02, 0.05]
filas_id3 = []
for ganancia in grid_gain:
    detalle = evaluate_temporal_cv(
        ID3(min_info_gain=ganancia), train, folds, 'discrete'
    )
    arbol_tmp = ID3(min_info_gain=ganancia).fit(X_train, train['winner'])
    filas_id3.append({
        'min_info_gain': ganancia,
        'error_validacion': detalle.validation_error.mean(),
        'error_validacion_std': detalle.validation_error.std(ddof=0),
        'macro_f1': detalle.macro_f1.mean(),
        'macro_f1_std': detalle.macro_f1.std(ddof=0),
        'error_train': detalle.train_error.mean(),
        'profundidad': arbol_tmp.get_depth(),
        'hojas': arbol_tmp.get_n_leaves(),
    })
cv_id3 = pd.DataFrame(filas_id3)
display(cv_id3.round(6))

fig, ejes = plt.subplots(1, 2, figsize=(10, 3.5), layout='constrained')
ejes[0].errorbar(cv_id3['min_info_gain'], cv_id3['error_validacion'],
                 yerr=cv_id3['error_validacion_std'], marker='o', capsize=3, label='Validación')
ejes[0].plot(cv_id3['min_info_gain'], cv_id3['error_train'],
             marker='s', linestyle='--', label='Train')
ejes[0].legend()
ejes[0].set(xlabel='min_info_gain', ylabel='error de validación',
            title='Error (1 - accuracy)')
ejes[1].errorbar(cv_id3['min_info_gain'], cv_id3['macro_f1'],
                 yerr=cv_id3['macro_f1_std'], marker='o', capsize=3)
ejes[1].set(xlabel='min_info_gain', ylabel='macro-F1',
            title='Macro-F1 de validación')
for eje in ejes:
    eje.grid(alpha=0.3)
plt.show()

id3_gain = float(
    cv_id3.sort_values(['macro_f1', 'min_info_gain'],
                       ascending=[False, True]).iloc[0]['min_info_gain']
)
fila_elegida = cv_id3[cv_id3['min_info_gain'] == id3_gain].iloc[0]
print('min_info_gain elegido:', id3_gain)
print(f'Con ese umbral el árbol tiene profundidad {fila_elegida["profundidad"]:.0f} '
      f'y {fila_elegida["hojas"]:.0f} hojas')

Vemos la misma medición de dos formas. El error de validación (1 - accuracy)
baja hasta `min_info_gain = 0.01` y después sube; el macro-F1 tiene su máximo en
`0.005` y cae desde `0.01`. Elegimos por macro-F1 porque no queremos un árbol que
ignore las clases minoritarias. Entre `0.0` y `0.005` la diferencia de macro-F1
(≈0.001) es mucho menor que la desviación entre folds (≈0.015), así que esos
valores no se distinguen con estos datos; `0.005` es el máximo nominal, y además
poda algo (589 hojas contra 644). A partir de `0.02` el árbol queda demasiado
podado y el rendimiento cae fuerte.

## 9. Naive Bayes propio

Naive Bayes combina la frecuencia de cada clase con la de sus atributos,
asumiendo independencia entre atributos dada la clase. Nuestra implementación
(`MEstimateCategoricalNB`) usa suavizado m-estimate:

$$
P(X_j = v \mid Y = c) = \frac{n_{jvc} + m \cdot p_{jv}}{n_c + m}
$$

con un prior uniforme de categorías `p_jv = 1 / K_j` (K_j = 4: tres bines más
el 0 reservado). `m` agrega conteos artificiales y evita probabilidades
cero; cuanto mayor es `m`, más se acerca a un prior uniforme y menos pesa la
frecuencia observada.

`CategoricalNB` de scikit-learn estima `(n + alpha) / (n_c + alpha * K)`, que
coincide con la nuestra tomando `alpha = m / 4`. Por eso barremos `m` sobre la
validación y repetimos la grilla con `CategoricalNB(m/4)`, confirmando la
equivalencia problema a problema.

Elegimos el `m` que maximiza el macro-F1 de validación, con el mismo desempate
que en ID3: ante valores que rinden igual, el menor `m` (no agregamos
suavizado artificial del que los datos no dan evidencia). Las columnas `*_std`
son la desviación estándar entre los tres folds y `error_validacion` es
`1 - accuracy` (sección 7).

In [ ]:
grid_m = [0.1, 1, 10, 100, 1000]
filas_nb = []
for m in grid_m:
    propio = evaluate_temporal_cv(
        MEstimateCategoricalNB(m=m, min_categories=4), train, folds, 'discrete')
    referencia = evaluate_temporal_cv(
        CategoricalNB(alpha=m / 4, min_categories=4, force_alpha=True),
        train, folds, 'discrete')
    filas_nb.append({
        'm': m,
        'alpha': m / 4,
        'error_validacion': propio.validation_error.mean(),
        'error_validacion_std': propio.validation_error.std(ddof=0),
        'macro_f1': propio.macro_f1.mean(),
        'macro_f1_sklearn': referencia.macro_f1.mean(),
        'error_train': propio.train_error.mean(),
        'error_validacion_sklearn': referencia.validation_error.mean(),
        'error_validacion_sklearn_std': referencia.validation_error.std(ddof=0),
        'error_train_sklearn': referencia.train_error.mean(),
        'macro_f1_std': propio.macro_f1.std(ddof=0),
    })
cv_nb = pd.DataFrame(filas_nb)
display(cv_nb.round(6))

fig, ejes = plt.subplots(1, 2, figsize=(10, 3.5), layout='constrained')
ejes[0].errorbar(cv_nb['m'], cv_nb['error_validacion'],
                 yerr=cv_nb['error_validacion_std'], marker='o', capsize=3, label='NB propio: validación')
ejes[0].errorbar(cv_nb['m'], cv_nb['error_validacion_sklearn'],
                 yerr=cv_nb['error_validacion_sklearn_std'], marker='x',
                 linestyle='--', capsize=3, label='sklearn CategoricalNB: validación')
ejes[0].plot(cv_nb['m'], cv_nb['error_train'], label='NB propio: train')
ejes[0].plot(cv_nb['m'], cv_nb['error_train_sklearn'], linestyle='--',
             label='sklearn CategoricalNB: train')
ejes[0].legend(fontsize=8)
ejes[0].set(xscale='log', xlabel='m', ylabel='error de validación',
            title='Error (1 - accuracy)')
ejes[1].errorbar(cv_nb['m'], cv_nb['macro_f1'],
                 yerr=cv_nb['macro_f1_std'], marker='o', capsize=3)
ejes[1].set(xscale='log', xlabel='m', ylabel='macro-F1',
            title='Macro-F1 de validación')
ejes[1].plot(cv_nb['m'], cv_nb['macro_f1_sklearn'], marker='x',
             linestyle='--', label='sklearn CategoricalNB')
ejes[1].lines[0].set_label('NB propio')
ejes[1].legend()
for eje in ejes:
    eje.grid(alpha=0.3)
plt.show()

nb_m = float(
    cv_nb.sort_values(['macro_f1', 'm'], ascending=[False, True]).iloc[0]['m']
)
print('m elegido:', nb_m, '-> alpha =', nb_m / 4)

De m=0.1 a 10 el error (0.5656) y el macro-F1 (0.3635) son idénticos: en estos folds las métricas de clasificación coinciden, aunque el suavizado sí altera las probabilidades. Desde m=100 el macro-F1 baja (0.3589, y 0.3342 con m=1000), consistente con que el suavizado empuja las probabilidades hacia la uniforme y aplana lo que dicen los datos, aunque esa caída es del mismo orden que la desviación entre folds (~0.022-0.033), así que no podemos afirmar con fuerza que m=100 ya sea peor que m=10. Solo m=1000 se aleja lo suficiente como para sospechar un efecto real. En cualquier caso, como 0.1, 1 y 10 empatan exactamente, el desempate de la sección 9 nos lleva a elegir el menor: m=0.1.

Las curvas de NB propio y sklearn CategoricalNB se superponen: comparten
entradas, dominio K=4 y suavizado equivalente alpha=m/4.

## 10. Random Forest: validación de las 12 configuraciones

Se conserva la grilla investigada: profundidades 4, 8 y sin límite; hojas
mínimas 50, 20, 5 y 1. Todas usan las seis tasas continuas, 300 árboles,
balanced_subsample y semilla 42. Seleccionamos por macro-F1 medio anual sin
redondear. En empate exacto se prefiere menor profundidad y luego mayor hoja,
según el orden declarado antes de medir. El error es 1 - accuracy.

In [ ]:
rf_grid = [
    {'max_depth': depth, 'min_samples_leaf': leaf}
    for depth in (4, 8, None) for leaf in (50, 20, 5, 1)
]
filas_rf = []
for tie_rank, params in enumerate(rf_grid):
    detalle = evaluate_temporal_cv(
        RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample',
                               random_state=RANDOM_STATE, n_jobs=2, **params),
        train, folds, 'continuous')
    filas_rf.append({
        'tie_rank': tie_rank,
        'profundidad': 'sin límite' if params['max_depth'] is None else str(params['max_depth']),
        'min_samples_leaf': params['min_samples_leaf'],
        'error_validacion': detalle.validation_error.mean(),
        'error_validacion_std': detalle.validation_error.std(ddof=0),
        'error_train': detalle.train_error.mean(),
        'macro_f1': detalle.macro_f1.mean(),
        'macro_f1_std': detalle.macro_f1.std(ddof=0),
    })
cv_rf = pd.DataFrame(filas_rf)
display(cv_rf.round(6))
rf_rank = int(cv_rf.sort_values(
    ['macro_f1', 'tie_rank'], ascending=[False, True], kind='stable'
).iloc[0]['tie_rank'])
rf_params = rf_grid[rf_rank].copy()
print('Random Forest elegido:', rf_params)

fig, ejes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
for depth, part in cv_rf.groupby('profundidad', sort=False):
    part = part.sort_values('min_samples_leaf')
    line = ejes[0].errorbar(part.min_samples_leaf, part.error_validacion,
                           yerr=part.error_validacion_std, marker='o', capsize=3,
                           label=f'Validación: profundidad {depth}')
    ejes[0].plot(part.min_samples_leaf, part.error_train, linestyle='--',
                 color=line.lines[0].get_color(), label=f'Train: profundidad {depth}')
    ejes[1].errorbar(part.min_samples_leaf, part.macro_f1,
                    yerr=part.macro_f1_std, marker='o', capsize=3, label=depth)
ejes[0].set(ylabel='Error (1 - accuracy)', title='Random Forest')
ejes[1].set(ylabel='Macro-F1 de validación', title='Random Forest')
for eje in ejes:
    eje.set_xlabel('min_samples_leaf')
    eje.grid(alpha=0.3)
    eje.legend(fontsize=7)
plt.show()

## 11. Comparación acotada de atributos

Con los hiperparámetros ya elegidos, comparamos únicamente las seis tasas y
esas mismas tasas más puntos y diferencia de gol recientes de ambos equipos
(diez entradas). ID3, NB propio y sklearn CategoricalNB usan los mismos folds
y la misma discretización: cortes fijos solo para las dos tasas last_5;
terciles aprendidos en el train del fold para el resto.

Seleccionamos cada modelo por macro-F1 medio sin redondear; ante empate se
prefieren seis entradas. No se reajustan hiperparámetros. Reutilizar los folds
para estas decisiones sucesivas puede introducir optimismo. Random Forest
conserva las seis tasas. La selección determina las columnas del ajuste final.

In [ ]:
feature_variants = {
    '6 tasas': list(NUMERIC_FEATURES),
    '10 atributos': list(NUMERIC_FEATURES) + [
        'home_points_per_match_5', 'away_points_per_match_5',
        'home_goal_diff_per_match_5', 'away_goal_diff_per_match_5',
    ],
}
feature_models = {
    'ID3': ID3(min_info_gain=id3_gain),
    'NB propio': MEstimateCategoricalNB(m=nb_m, min_categories=4),
    'sklearn CategoricalNB': CategoricalNB(
        alpha=nb_m / 4, min_categories=4, force_alpha=True),
}
filas_atributos = []
for nombre, modelo in feature_models.items():
    for rank, (variante, columnas) in enumerate(feature_variants.items()):
        detalle = evaluate_temporal_cv(
            modelo, train, folds, 'discrete', feature_columns=columnas)
        filas_atributos.append({
            'modelo': nombre, 'variante': variante,
            'n_features': len(columnas), 'tie_rank': rank,
            'macro_f1': detalle.macro_f1.mean(),
            'error_validacion': detalle.validation_error.mean(),
            'error_validacion_std': detalle.validation_error.std(ddof=0),
            'error_train': detalle.train_error.mean(),
        })
comparacion_cols = pd.DataFrame(filas_atributos)
display(comparacion_cols.round(6))


def select_feature_columns(comparison, variants):
    ranked = comparison.sort_values(
        ['macro_f1', 'n_features', 'tie_rank', 'modelo'],
        ascending=[False, True, True, True], kind='stable')
    return {
        row['modelo']: list(variants[row['variante']])
        for _, row in ranked.groupby('modelo', sort=False).head(1).iterrows()
    }


selected_columns = select_feature_columns(comparacion_cols, feature_variants)
id3_columns = selected_columns['ID3']
nb_columns = selected_columns['NB propio']
nb_ref_columns = selected_columns['sklearn CategoricalNB']
display(comparacion_cols.sort_values(
    ['macro_f1', 'n_features', 'tie_rank', 'modelo'],
    ascending=[False, True, True, True], kind='stable'
).groupby('modelo', sort=False).head(1))
print('Hiperparámetros seleccionados:', {
    'ID3 min_info_gain': id3_gain, 'NB m': nb_m,
    'sklearn NB alpha': nb_m / 4, 'Random Forest': rf_params})

## 12. Ajuste final con las configuraciones seleccionadas

Los modelos se ajustan con todo el train hasta 2023. Cada discretizador aprende
sus cuantiles solo de ese train y recibe las columnas elegidas por validación.
El bosque usa exactamente la configuración seleccionada de la grilla.

Para reproducir la evaluación final del informe, usar RUN_FINAL_TEST=True.
RUN_FINAL_TEST=False permite ejecutar solo selección y ajuste. Las métricas
vigentes corresponden a 14.705 partidos de train y 472 de evaluación.

In [ ]:
discretizer = new_discretizer(id3_columns).fit(train[id3_columns])
X_train = discretizer.transform(train[id3_columns])
arbol_id3 = ID3(min_info_gain=id3_gain).fit(X_train, train['winner'])

disc_final = new_discretizer(nb_columns).fit(train[nb_columns])
Xnb_train = disc_final.transform(train[nb_columns])
nb_final = MEstimateCategoricalNB(m=nb_m, min_categories=4).fit(
    Xnb_train, train['winner'])
disc_ref = new_discretizer(nb_ref_columns).fit(train[nb_ref_columns])
nb_ref = CategoricalNB(alpha=nb_m / 4, min_categories=4, force_alpha=True).fit(
    disc_ref.transform(train[nb_ref_columns]), train['winner'])

X_train_raw = train[NUMERIC_FEATURES].to_numpy()
y_train = train['winner'].to_numpy()
competidores = {
    'sklearn DT (mismo input)': DecisionTreeClassifier(
        criterion='entropy', random_state=RANDOM_STATE),
    'sklearn DT (tasas crudas)': DecisionTreeClassifier(
        criterion='entropy', random_state=RANDOM_STATE),
    'sklearn RandomForest (tasas crudas)': RandomForestClassifier(
        n_estimators=300, class_weight='balanced_subsample',
        random_state=RANDOM_STATE, n_jobs=2, **rf_params),
}
for nombre, modelo in competidores.items():
    modelo.fit(X_train if 'mismo input' in nombre else X_train_raw, y_train)
base_10 = TenYearWinRateClassifier().fit(train[BASELINE_FEATURES], train['winner'])
print('Ajuste completado solo con train; evaluación de test pendiente.')

## 13. ID3: entrenar y medir sobre el test

Con el umbral y los atributos elegidos, el árbol ya está ajustado con todo
el train. Esta evaluación de 2024-2025 requiere RUN_FINAL_TEST=True. `classification_report` da, para cada clase,
**precision**, **recall**, **F1** y **support**, y dos promedios (macro y
weighted); la **matriz de confusion** los muestra por clase: filas = real,
columnas = predicho. También reportamos la importancia de cada atributo
(ganancia acumulada normalizada) para ver que usa el árbol.

In [ ]:
if RUN_FINAL_TEST:
    X_test = discretizer.transform(test[id3_columns])
    y_test = test['winner'].to_numpy()
    y_pred = arbol_id3.predict(X_test)

    print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
    print('Macro-F1:', round(
        f1_score(y_test, y_pred, average='macro', zero_division=0), 4))
    print('Profundidad:', arbol_id3.get_depth(), '| hojas:', arbol_id3.get_n_leaves())
    reporte_por_clases('ID3 propio', y_test, y_pred)

    importancias = pd.DataFrame({
        'atributo': id3_columns,
        'importancia': arbol_id3.feature_importances_,
    }).sort_values('importancia', ascending=False)
    display(importancias)

    matriz = confusion_matrix(y_test, y_pred, labels=CLASSES)
    fig, axis = plt.subplots(figsize=(4.5, 4))
    axis.imshow(matriz, cmap='Blues')
    axis.set(xticks=range(3), xticklabels=CLASSES,
             yticks=range(3), yticklabels=CLASSES,
             xlabel='predicción', ylabel='real')
    for fila in range(3):
            for col in range(3):
                axis.text(col, fila, str(matriz[fila, col]),
                          ha='center', va='center')
    plt.show()
else:
    print('Evaluación final pendiente: RUN_FINAL_TEST=False.')

Con RUN_FINAL_TEST=True se reproduce el ID3 final: accuracy 0.4492 y
macro-F1 0.4238, usando diez atributos y min_info_gain=0.005.

## 14. Ejemplos: donde acierta y donde falla

Vemos partidos reales de test con la predicción. La columna `acierta` indica
si el árbol acerto. Los empates son la clase que más se pierde.

In [ ]:
if RUN_FINAL_TEST:
    predicciones = test[['date', 'home', 'away', 'winner']].copy()
    predicciones['prediccion_id3'] = y_pred
    predicciones['acierta'] = predicciones['prediccion_id3'] == predicciones['winner']
    display(predicciones.sample(8, random_state=RANDOM_STATE))
    print('Aciertos totales:', int(predicciones['acierta'].sum()), '/', len(predicciones))

    empates_reales = predicciones[predicciones['winner'] == 'E']
    print('Sobre los', len(empates_reales), 'empates reales del test, '
          f'ID3 predijo empate en '
          f'{(empates_reales["prediccion_id3"] == "E").sum()} '
          f'({(empates_reales["prediccion_id3"] == "E").mean():.1%}).')
    display(empates_reales.head(3))
else:
    print('Evaluación final pendiente: RUN_FINAL_TEST=False.')

### 14.1. Predicción de un partido nuevo

Para demostrar que un modelo entrenado sirve para clasificar una instancia
nueva, `predict_new_match` (definida en src/features.py) aplica a un partido
hipotetico exactamente el mismo calculo de atributos del resto del notebook:
solo partidos anteriores, los mismos calculos y la misma imputación neutra si
no hay historial. No se reimplementan tasas ni se le pasa ningun resultado.

Tomamos tres partidos futuros (no estan en el dataset) y los clasificamos con
dos modelos ya entrenados: el ID3 propio (discretizado) y el clasificador base
(tasas crudas). No buscamos optimizar nada, solo mostrar el uso.

In [ ]:
if RUN_FINAL_TEST:
    base_demo = TenYearWinRateClassifier().fit(
        train[BASELINE_FEATURES], train['winner'])

    partidos_nuevos = [
        ('Nacional', 'CA Penarol', '2025-11-16'),
        ('Boston River', 'Torque FC', '2025-11-16'),
        ('CA Juventud', 'Plaza Colonia', '2025-11-16'),
    ]

    filas_demo = []
    for home, away, fecha in partidos_nuevos:
        id3_demo = predict_new_match(
            home, away, fecha, matches, arbol_id3,
            feature_columns=id3_columns, discretizer=discretizer)
        base_demo_pred = predict_new_match(
            home, away, fecha, matches, base_demo,
            feature_columns=BASELINE_FEATURES, discretizer=None)
        filas_demo.append({
            'fecha': fecha,
            'partido': f'{home} vs {away}',
            'forma local': round(id3_demo['features']['home_win_rate_last_5'], 2),
            'forma visitante': round(id3_demo['features']['away_win_rate_last_5'], 2),
            'h2h local': round(id3_demo['features']['home_win_rate_h2h_as_home'], 2),
            'ID3': id3_demo['prediccion'],
            'Base 10 años': base_demo_pred['prediccion'],
        })
    display(pd.DataFrame(filas_demo))
else:
    print('Evaluación final pendiente: RUN_FINAL_TEST=False.')

## 15. Comparadores

Comparamos el ID3 propio con:

- `sklearn DecisionTree` con el mismo input discretizado: sirve para comparar
  nuestra implementación con la de la biblioteca, con el mismo input para las
  dos.
- `sklearn DecisionTree` con las tasas crudas: puede cortar en cualquier
  umbral, muestra cuanto gana un árbol sobre el input continuo.
- `sklearn RandomForest` (300 árboles, `class_weight='balanced_subsample'`):
  el bosque da más peso a las clases chicas en cada árbol para no ignorar el
  empate. Es una referencia alta, no una competencia directa.
- El clasificador base: gana el equipo con mayor proporción de victorias en
  los últimos diez años (una regla fija, sin aprendizaje).

Todos se entrenan solo con train y se evalúan sobre el mismo test de
2024-2025; la tabla compara accuracy y macro-F1 (sección 7).

El bosque usa la profundidad y hoja mínima elegidas en la sección 10.
La evaluación está desactivada por defecto.

In [ ]:
if RUN_FINAL_TEST:
    X_test_raw = test[NUMERIC_FEATURES].to_numpy()

    filas_comp = []
    predicciones_comp = {}
    for nombre, modelo in competidores.items():
        usa_codigos = 'mismo input' in nombre
        X_entrena = X_train if usa_codigos else X_train_raw
        X_prueba = X_test if usa_codigos else X_test_raw
        predicha = modelo.predict(X_prueba)
        predicciones_comp[nombre] = predicha
        filas_comp.append({
            'modelo': nombre,
            'accuracy': round(accuracy_score(y_test, predicha), 4),
            'macro_f1': round(
                f1_score(y_test, predicha, average='macro', zero_division=0), 4),
        })

    pred_base = base_10.predict(test[BASELINE_FEATURES])
    filas_comp.append({
        'modelo': 'Base 10 años (clasificador base)',
        'accuracy': round(accuracy_score(y_test, pred_base), 4),
        'macro_f1': round(
            f1_score(y_test, pred_base, average='macro', zero_division=0), 4),
    })
    filas_comp.append({
        'modelo': 'ID3 propio',
        'accuracy': round(accuracy_score(y_test, y_pred), 4),
        'macro_f1': round(
            f1_score(y_test, y_pred, average='macro', zero_division=0), 4),
    })
    display(pd.DataFrame(filas_comp))
else:
    print('Evaluación final pendiente: RUN_FINAL_TEST=False.')

## 16. Naive Bayes: evaluación final

Ambos NB ya se ajustaron usando las columnas elegidas por validación y el mismo
preprocesado de la búsqueda. Este bloque solo evalúa si RUN_FINAL_TEST=True.

In [ ]:
if RUN_FINAL_TEST:
    Xnb_test = disc_final.transform(test[nb_columns])
    Xnb_ref_test = disc_ref.transform(test[nb_ref_columns])
    pred_nb = nb_final.predict(Xnb_test)
    pred_ref = nb_ref.predict(Xnb_ref_test)
    print('Same predictions (propio vs CategoricalNB):',
          bool((pred_nb == pred_ref).all()))
    print('Accuracy NB:', round(accuracy_score(y_test, pred_nb), 4))
    print('Macro-F1 NB:', round(
        f1_score(y_test, pred_nb, average='macro', zero_division=0), 4))
    reporte_por_clases(f'NB propio ({len(nb_columns)} atributos)', y_test, pred_nb)

    fig, ejes = plt.subplots(1, 2, figsize=(9, 4), layout='constrained')
    for eje, (nombre, pred) in zip(ejes, [('NB final', pred_nb),
                                          ('Base 10 años', pred_base)]):
        matriz = confusion_matrix(y_test, pred, labels=CLASSES)
        eje.imshow(matriz, cmap='Blues')
        eje.set(xticks=range(3), xticklabels=CLASSES,
                yticks=range(3), yticklabels=CLASSES,
                xlabel='predicción', ylabel='real', title=nombre)
        for fila in range(3):
            for col in range(3):
                eje.text(col, fila, str(matriz[fila, col]),
                         ha='center', va='center')
    plt.show()
else:
    print('Evaluación final pendiente: RUN_FINAL_TEST=False.')

Los NB finales coinciden en las 472 etiquetas: accuracy 0.4788 y macro-F1
0.4454. El baseline obtiene 0.4619 y 0.3564, respectivamente.

## 17. Efectividad del algoritmo

Juntamos todo en una tabla sobre el mismo test y lo comparamos con la regla
trivial de predecir siempre L (la clase mayoritaria). Además vemos cuántos
empates captura cada modelo.

In [ ]:
if RUN_FINAL_TEST:
    siempre_l = np.full(len(y_test), 'L')
    filas_resumen = filas_comp + [
        {'modelo': f'NB propio ({len(nb_columns)} atributos)',
         'accuracy': round(accuracy_score(y_test, pred_nb), 4),
         'macro_f1': round(
             f1_score(y_test, pred_nb, average='macro', zero_division=0), 4)},
        {'modelo': 'Siempre L',
         'accuracy': round(accuracy_score(y_test, siempre_l), 4),
         'macro_f1': round(
             f1_score(y_test, siempre_l, average='macro', zero_division=0), 4)},
    ]
    resumen = pd.DataFrame(filas_resumen)
    display(resumen.sort_values('accuracy', ascending=False).reset_index(drop=True))

    print('Empates reales en test:', int((y_test == 'E').sum()),
          'de', len(y_test))
    for nombre, pred in [('ID3', y_pred), ('NB final', pred_nb),
                         ('Base 10 años', pred_base)]:
        print(f'{nombre:<15} predijo E en '
              f'{int((pred == "E").sum())} partidos '
              f'(acierto {(pred[y_test == "E"] == "E").mean():.1%} de los reales)')
else:
    print('Evaluación final pendiente: RUN_FINAL_TEST=False.')

### 17.1. Métricas por clase de los cinco modelos

La misma función `reporte_por_clases` mide a los cinco modelos sobre el mismo
test. Ya la usamos arriba para ID3 y NB propio; la repetimos acá con todos los
comparadores para ver de un vistazo como rinde cada clase (definiciones en la
sección 7).

In [ ]:
if RUN_FINAL_TEST:
    reporte_por_clases('Base 10 años', y_test, pred_base)
    reporte_por_clases('sklearn DT (mismo input)', y_test,
                       predicciones_comp['sklearn DT (mismo input)'])
    reporte_por_clases('sklearn RandomForest (tasas crudas)', y_test,
                       predicciones_comp['sklearn RandomForest (tasas crudas)'])
    reporte_por_clases('sklearn CategoricalNB', y_test, pred_ref)

    # Tablas y diagnóstico ya retenidos en el informe; sin nuevos ajustes.
    predicciones_informe = {
        'ID3': y_pred, 'NB propio': pred_nb, 'CategoricalNB': pred_ref,
        'Random Forest': predicciones_comp['sklearn RandomForest (tasas crudas)'],
        'Base 10 años': pred_base,
    }
    metricas_informe = pd.DataFrame([
        {'model': nombre, 'n_test': len(y_test),
         'correct': int((pred == y_test).sum()),
         'accuracy': accuracy_score(y_test, pred),
         'macro_f1': f1_score(y_test, pred, labels=CLASSES,
                              average='macro', zero_division=0)}
        for nombre, pred in predicciones_informe.items()
    ])
    display(metricas_informe)
    for nombre, pred in predicciones_informe.items():
        print(nombre, '(filas reales, columnas predichas: E/L/V)')
        display(pd.DataFrame(confusion_matrix(y_test, pred, labels=CLASSES),
                             index=CLASSES, columns=CLASSES))
    print('Máxima diferencia absoluta de probabilidades NB:',
          np.max(np.abs(nb_final.predict_proba(Xnb_test)
                        - nb_ref.predict_proba(Xnb_ref_test))))

    ejemplos_informe = test[['date', 'home', 'away', 'winner']].copy()
    ejemplos_informe['prediction_nb_propio'] = pred_nb
    ejemplos_informe = ejemplos_informe.merge(
        matches[['date', 'home', 'away', 'gh', 'ga']],
        on=['date', 'home', 'away'], validate='one_to_one')
    ejemplos_informe['score'] = (
        ejemplos_informe['gh'].astype(str) + '-' + ejemplos_informe['ga'].astype(str))
    ejemplos_informe = (
        ejemplos_informe.sort_values(['date', 'home', 'away'])
        .groupby(['winner', 'prediction_nb_propio'], sort=True, as_index=False)
        .head(1).sort_values(['winner', 'prediction_nb_propio']).reset_index(drop=True))
    display(ejemplos_informe)

    senal = np.select(
        [test['home_points_per_match_5'] > test['away_points_per_match_5'],
         test['home_points_per_match_5'] < test['away_points_per_match_5']],
        ['Local > visitante', 'Local < visitante'], default='Local = visitante')
    filas_escenarios = []
    for grupo in ('Local > visitante', 'Local = visitante', 'Local < visitante'):
        mascara = senal == grupo
        for nombre, pred in predicciones_informe.items():
            filas_escenarios.append({
                'subgroup': grupo, 'model': nombre, 'n_test': int(mascara.sum()),
                'actual_E_L_V': '/'.join(str(int((y_test[mascara] == c).sum()))
                                         for c in CLASSES),
                'correct': int((pred[mascara] == y_test[mascara]).sum()),
                'accuracy': accuracy_score(y_test[mascara], pred[mascara]),
                'macro_f1': f1_score(y_test[mascara], pred[mascara], labels=CLASSES,
                                     average='macro', zero_division=0),
            })
    escenarios_informe = pd.DataFrame(filas_escenarios)
    display(escenarios_informe)
else:
    print('Evaluación omitida: RUN_FINAL_TEST=False; usar True para reproducir el informe.')


### Interpretación de los resultados del informe

NB supera al baseline en accuracy y macro-F1; ID3 y Random Forest solo lo
superan en macro-F1. E tiene el menor F1 en todos los modelos. Los ejemplos
anteriores toman el primer partido de cada combinación real/predicha de NB,
ordenado por fecha y equipos. Los escenarios agrupan por diferencia de puntos
recientes; son descriptivos y no demuestran efectos causales del atributo.
Los modelos y cortes permanecen fijos durante test; los históricos incorporan
solo fechas anteriores. El test ya inspeccionado y la reutilización de folds
limitan la interpretación como evidencia independiente.
